# Course notebook — Foundations of Intelligence

Understand intelligence as search, learning, representation, decision-making, and feedback before studying language models.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Goals and representations


### Lecture 01 · What Intelligence Means

Treat intelligence as a collection of capabilities rather than a mystical substance. Separate perception, representation, memory, learning, reasoning, planning, action, and adaptation.

**Predict before running.** Intelligence is system behavior under goals and constraints, not a property inferred from fluent language. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Create a capability map for a familiar human task.


In [ ]:
"""Lab 1.1: What Intelligence Means"""

GOAL = "route P1 incidents to on-call"
CAPABILITIES = ["perceive", "represent", "decide", "act"]
ticket = "All regions down — writes failing"
features = set(ticket.lower().split())
severity = "P1" if {"down", "failing"} & features else "P2"
trace = {cap: cap for cap in CAPABILITIES}
trace["represent"] = sorted(features)
trace["decide"] = severity
print({"goal": GOAL, "trace": trace})


### Lecture 02 · From Symbols to Statistics

Trace the path from symbolic rules and expert systems to statistical learning, deep learning, foundation models, and agents. Each era solved different problems and retained useful ideas.

**Predict before running.** New paradigms usually absorb rather than erase earlier engineering techniques. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement a tiny rule engine and document where it becomes brittle.


In [ ]:
"""Lab 1.2: From Symbols to Statistics"""

RULES = [("urgent", "P1"), ("question", "P3")]
tickets = [
    "URGENT: payment gateway offline",
    "urgent feature request for dashboard",
    "question about invoice format",
]
def rule_route(text):
    lower = text.lower()
    for kw, sev in RULES:
        if kw in lower:
            return sev
    return "P2"
for t in tickets:
    print({"ticket": t[:40], "route": rule_route(t)})


**Section checkpoint.** Write a support-routing goal and identify an example where a fixed keyword rule misroutes a request.


## Search and mathematics


### Lecture 03 · Search, Planning, and Decisions

See classical search as an explicit form of reasoning. Learn how state, actions, transition models, costs, heuristics, and stopping rules turn a vague goal into an algorithm.

**Predict before running.** Reasoning can be viewed as controlled search over possible states or candidate solutions. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement breadth-first search and A* on the same maze.


In [ ]:
"""Lab 1.3: Search, Planning, and Decisions"""

GRAPH = {"start": ["oncall_a", "oncall_b"], "oncall_a": ["lead"], "oncall_b": ["lead"], "lead": []}
GOAL = "lead"
def bfs(start, goal):
    queue = [(start, [start])]
    seen = {start}
    while queue:
        node, path = queue.pop(0)
        if node == goal:
            return path
        for nxt in GRAPH.get(node, []):
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, path + [nxt]))
    return None
print("escalation path:", bfs("start", GOAL))


### Lecture 04 · The Mathematics Engineers Need

Develop intuition for vectors, matrices, probability, distributions, statistics, entropy, gradients, and optimization without turning the book into a mathematics degree.

**Predict before running.** Mathematics is a compact language for relationships, uncertainty, and change. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Compute dot products, cosine similarity, softmax, and one gradient update by hand.


In [ ]:
"""Lab 1.4: The Mathematics Engineers Need"""

import math
a = [3.0, 0.0, 1.0]
b = [2.0, 0.0, 2.0]
def cosine(u, v):
    dot = sum(x*y for x, y in zip(u, v))
    nu = math.sqrt(sum(x*x for x in u))
    nv = math.sqrt(sum(y*y for y in v))
    return dot / (nu * nv)
def softmax(xs):
    m = max(xs)
    ex = [math.exp(x - m) for x in xs]
    s = sum(ex)
    return [e/s for e in ex]
sims = [cosine(a, b), cosine(a, a), cosine(b, b)]
print("cosines:", [round(c, 3) for c in sims])
print("softmax:", [round(p, 3) for p in softmax(sims)])


**Section checkpoint.** Trace BFS and A* on the same grid. Record both path length and states expanded.


## Learning and decisions


### Lecture 05 · Learning and Generalization

Distinguish memorization from generalization and training from inference. Understand data-generating processes, inductive bias, overfitting, underfitting, and distribution shift.

**Predict before running.** A system is useful when it performs under future conditions, not merely on its training examples. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Fit increasingly flexible models to a small noisy dataset and plot errors.


In [ ]:
"""Lab 1.5: Learning and Generalization"""

data = [(1, 0), (2, 0), (3, 1), (4, 1), (5, 2)]
labels = [0, 0, 1, 1, 0]
def mse_line(m, b, pts):
    return sum((m*x + b - y)**2 for x, y in pts) / len(pts)
best = min(((m, b, mse_line(m, b, data)) for m in [0, 0.5, 1.0] for b in [0, 0.5]), key=lambda t: t[2])
holdout = [(6, 1), (7, 1)]
test_err = mse_line(best[0], best[1], holdout)
print({"fit": best[:2], "train_mse": round(best[2], 3), "holdout_mse": round(test_err, 3)})


### Lecture 06 · Engineering with Uncertainty

Translate uncertain predictions into decisions with thresholds, costs, calibration, fallback behavior, and human oversight.

**Predict before running.** Prediction and decision are separate layers; consequences belong in the decision layer. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Design a decision policy for a high-cost false-positive scenario.


In [ ]:
"""Lab 1.6: Engineering with Uncertainty"""

scores = [0.92, 0.61, 0.48, 0.33]
COST = {"fp": 200, "fn": 50000}
def expected_cost(threshold):
    decisions = [s >= threshold for s in scores]
    truth = [True, True, False, False]
    fp = sum(d and not t for d, t in zip(decisions, truth))
    fn = sum(not d and t for d, t in zip(decisions, truth))
    return fp * COST["fp"] + fn * COST["fn"]
for t in [0.5, 0.6, 0.7, 0.8]:
    print(f"threshold={t} expected_cost={expected_cost(t)}")


**Section checkpoint.** Assign separate costs to a wrong route and a human review. Specify when review becomes cheaper.


## Course workshop

Cover the route with a blocked cell and compare the path returned by each search. Record when the heuristic saves work.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Compare breadth-first search and A* on the same grid."""
from collections import deque
from heapq import heappop, heappush

GRID = ["S...#", ".##..", "...#.", ".#...", "...G."]
START, GOAL = (0, 0), (4, 3)


def neighbors(node):
    r, c = node
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(GRID) and 0 <= nc < len(GRID[0]) and GRID[nr][nc] != "#":
            yield nr, nc


def reconstruct(parent, node):
    path = [node]
    while node in parent:
        node = parent[node]
        path.append(node)
    return list(reversed(path))


def bfs():
    queue, parent, seen = deque([START]), {}, {START}
    expanded = 0
    while queue:
        node = queue.popleft(); expanded += 1
        if node == GOAL:
            return reconstruct(parent, node), expanded
        for nxt in neighbors(node):
            if nxt not in seen:
                seen.add(nxt); parent[nxt] = node; queue.append(nxt)


def astar():
    frontier, parent, cost = [(0, START)], {}, {START: 0}
    expanded = 0
    while frontier:
        _, node = heappop(frontier); expanded += 1
        if node == GOAL:
            return reconstruct(parent, node), expanded
        for nxt in neighbors(node):
            new_cost = cost[node] + 1
            if nxt not in cost or new_cost < cost[nxt]:
                cost[nxt] = new_cost; parent[nxt] = node
                heuristic = abs(nxt[0] - GOAL[0]) + abs(nxt[1] - GOAL[1])
                heappush(frontier, (new_cost + heuristic, nxt))


if __name__ == "__main__":
    for name, search in (("BFS", bfs), ("A*", astar)):
        path, expanded = search()
        print(f"{name}: path_length={len(path)-1}, expanded={expanded}, path={path}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
bfs_path, bfs_expanded = bfs()
astar_path, astar_expanded = astar()
assert len(bfs_path) == len(astar_path)
assert astar_expanded <= bfs_expanded
print('Workshop checks passed')


## Course project

Build and compare a rule-based solver, a search-based solver, and a learned predictor for one bounded problem.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
